# Bayesian Optimisation — Function 3 (3D)

**Author:** Dibyajyoti Pradhan  
**Programme:** Imperial College London Professional Certificate in ML/AI  
**Module:** BBO Capstone (Modules 12–22)  

**Strategy note:** Gradual improvement along consistent trajectory.

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add src/ to path for reusable utilities
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
from bbo_utils import (
    lhs_candidates, fit_gp, gp_posterior,
    ucb_acquisition, suggest_next_query,
    dimension_sensitivity, plot_running_best,
)

np.random.seed(42)

## 2. Load Initial Data

Initial observations provided at the start of the BBO challenge.

In [ ]:
data_dir = os.path.join('..', 'data', 'initial_data', 'function_3')

X_init = np.load(os.path.join(data_dir, 'initial_inputs.npy'))
y_init = np.load(os.path.join(data_dir, 'initial_outputs.npy'))

print(f'Initial inputs shape:  {X_init.shape}')
print(f'Initial outputs shape: {y_init.shape}')
print(f'Output range: [{y_init.min():.6f}, {y_init.max():.6f}]')
print(f'Best initial output:   {y_init.max():.6f}')
print(f'Best initial input:    {X_init[np.argmax(y_init)]}')

## 3. Add Subsequent Query Data

Queries from Weeks 1–5 with confirmed outputs.  
Week 6 submitted — awaiting results (not included in training).

In [ ]:
# Weekly queries — Weeks 1–5 (confirmed outputs)
queries_X = np.array([
    [0.437056, 0.996752, 0.890886],   # W1
    [0.488361, 0.663509, 0.183247],   # W2
    [0.267336, 0.226569, 0.437517],   # W3
    [0.222230, 0.333831, 0.468755],   # W4
    [0.276762, 0.350396, 0.433114],   # W5
])
queries_y = np.array([
    -0.091753256062798,    # W1
    -0.116533729497695,    # W2
    -0.036082074169511,    # W3
    -0.008519051360942,    # W4
    -0.022909225627902,    # W5
])

# Week 6 submitted — awaiting result (excluded from training)
# x_w6 = np.array([0.213857, 0.300587, 0.437974])

X_train = np.vstack([X_init, queries_X])
y_train = np.append(y_init, queries_y)

print(f'Total observations: {len(y_train)}')
print(f'Best observed output: {y_train.max():.6f}')
print(f'Best observed input:  {X_train[np.argmax(y_train)]}')

## 4. Data Exploration

In [ ]:
print('Output statistics:')
print(f'  Min:  {y_train.min():.6f}')
print(f'  Max:  {y_train.max():.6f}')
print(f'  Mean: {y_train.mean():.6f}')
print(f'  Std:  {y_train.std():.6f}')

# Running best across all observations
plot_running_best(y_train, title='Function 3 — Running Best Output')

## 5. Bayesian Optimisation (UCB, β=0.5)

Strategy: exploitation-heavy (trust region ±0.06)

In [ ]:
next_query, fitted_model = suggest_next_query(
    X_train=X_train,
    y_train=y_train,
    n_candidates=1_000_000,
    beta=0.5,
    length_scale=0.1,
    noise_level=1e-6,
    fit_noise=True,
    trust_region_radius=0.06,
    seed=42,
)

# Format to 6 decimal places (BBO portal requirement)
coords = ', '.join(f'{v:.6f}' for v in next_query)
print(f'Week 7 suggested query for Function 3:')
print(f'  ({coords})')

## 6. Week 7 Strategy Summary

| Field | Value |
|-------|-------|
| Function | F3 (3D) |
| Total observations | n/a (computed above) |
| Acquisition | UCB, β=0.5 |
| Trust region | ±0.06 per dim |
| Strategy | Best at W4 (-0.0085). W5 slightly regressed. Refine near [0.22, 0.33, 0.47]. |

> Submit the coordinates printed above via the BBO portal. All inputs must be in `[0, 1)` to 6 decimal places.